# 30B — Source-balance v2 dense review and replacement screen

**CPU → Run all.** This follows the verified 30A screen. It performs two evidence-only jobs in one run: (1) creates 2-FPS sheets for 17 pinned row/sit-up/pull-up locators that passed the first screen, and (2) creates 5-second full-timeline sheets for five replacement sit-up/lat-pulldown candidates. It reuses a valid 360p cache when available and records individual source failures without aborting the batch. No raw video, landmarks, labels, or training data are exported; every source remains `training_allowed=false`.


In [ ]:
%pip install -q --upgrade "yt-dlp[default]" opencv-python-headless pandas==2.2.3


In [ ]:
from pathlib import Path
import hashlib,json,platform,re,shutil,subprocess,urllib.request,zipfile
import cv2,numpy as np,pandas as pd
from google.colab import files
from IPython.display import display

PINNED_COMMIT='93e366bc574b886bff33c29b880ef1051fb11f10'
RAW=f'https://raw.githubusercontent.com/bariscelikk1/cALorie/{PINNED_COMMIT}/'
INPUTS={
 'ml/data/source_balance_v2/candidates.csv':'d69bdff1aed46349f67b695676debe6c767d11f3cda62844251084f9eca0aa17',
 'ml/data/source_balance_v2/supplemental_candidates.csv':'a550f6856fc50afa889ebe0501e2d5e335a67ceaf60bc241d9929c024b170221',
 'ml/data/source_balance_v2/screening_review.csv':'0be996b2092276d84ad2296de07f66378accaba64407c261dad8c2438006caec',
 'ml/data/source_balance_v2/dense_review_targets.csv':'846692d71451eade130c3e1014a070bea12dac507f16e3e40993fc849175abca',
 'ml/data/youtube_creator_benchmark_v1/candidates.csv':'aa3063810fa3b0e9fe169c04281ef65cc33a442d18be0fa0e5fec7669e06268b',
 'ml/data/youtube_creator_benchmark_v1/excluded_source_inventory.csv':'ecf18c29ef5b7ba1eca3f45677e463df256b965888f3913ae55606807baf2232',
 'ml/data/youtube_mixed_workouts_v1/manifest.csv':'1d58ed25261a172e49abe3f9b7313236d339e5fbe067ecccf838c6f187efb3eb',
 'ml/data/youtube_long_workouts_v2/candidates.csv':'fb8af2597dfb93942bb2512e524352036ac6ea5e6ea2df37c1ddd29c0c78da34',
}
WORK=Path('/content/calorie_source_balance_v2_work')
OUTPUT=Path('/content/calorie_source_balance_v2_dense_and_replacements')
for folder in (WORK,WORK/'videos',WORK/'dense_frames',WORK/'screen_frames',OUTPUT,OUTPUT/'dense_contact_sheets',OUTPUT/'replacement_contact_sheets',OUTPUT/'metadata'):
 folder.mkdir(parents=True,exist_ok=True)

def sha256(path):
 d=hashlib.sha256()
 with Path(path).open('rb') as f:
  for block in iter(lambda:f.read(1024*1024),b''):d.update(block)
 return d.hexdigest()

def fetch(relative,digest):
 path=WORK/'pinned_inputs'/relative
 path.parent.mkdir(parents=True,exist_ok=True)
 if not path.exists() or sha256(path)!=digest:
  tmp=path.with_suffix(path.suffix+'.part');tmp.unlink(missing_ok=True)
  urllib.request.urlretrieve(RAW+relative,tmp)
  assert sha256(tmp)==digest,relative
  tmp.replace(path)
 return path

paths={name:fetch(name,digest) for name,digest in INPUTS.items()}
assert len(set(paths.values()))==len(INPUTS)
assert all(sha256(paths[name])==digest for name,digest in INPUTS.items())
main=pd.read_csv(paths['ml/data/source_balance_v2/candidates.csv'])
supplemental=pd.read_csv(paths['ml/data/source_balance_v2/supplemental_candidates.csv'])
screening=pd.read_csv(paths['ml/data/source_balance_v2/screening_review.csv'])
targets=pd.read_csv(paths['ml/data/source_balance_v2/dense_review_targets.csv'])
reserved=pd.concat([pd.read_csv(paths[name]) for name in (
 'ml/data/youtube_creator_benchmark_v1/candidates.csv',
 'ml/data/youtube_creator_benchmark_v1/excluded_source_inventory.csv',
 'ml/data/youtube_mixed_workouts_v1/manifest.csv',
 'ml/data/youtube_long_workouts_v2/candidates.csv')],ignore_index=True)
normalize=lambda value:re.sub('[^a-z0-9]+','',str(value).casefold())
all_intake=pd.concat([main,supplemental],ignore_index=True)
assert all_intake.video_id.is_unique and all_intake.channel_id.is_unique and all_intake.creator.map(normalize).is_unique
assert set(all_intake.video_id).isdisjoint(set(reserved.video_id))
assert set(all_intake.creator.map(normalize)).isdisjoint(set(reserved.creator.map(normalize)))
assert (all_intake.status=='metadata_only').all() and (~all_intake.training_allowed).all()
assert targets.review_id.is_unique and set(targets.video_id).issubset(set(main.video_id))
assert (targets.end_s>targets.start_s).all() and (~targets.training_allowed).all()
advanced=set(screening.loc[screening.screening_decision=='advance_dense_review','video_id'])
assert set(targets.video_id).issubset(advanced)
selected_ids=set(targets.video_id)|set(supplemental.video_id)
selected=all_intake[all_intake.video_id.isin(selected_ids)].reset_index(drop=True)
assert len(selected)==len(selected_ids)
print('CPU 30B:',len(targets),'dense regions and',len(supplemental),'replacement sources across',len(selected),'videos; training forbidden')
display(selected[['creator','target_class','video_id','role']])


In [ ]:
def deno_args():
 asset={'x86_64':'deno-x86_64-unknown-linux-gnu.zip','aarch64':'deno-aarch64-unknown-linux-gnu.zip'}.get(platform.machine().lower())
 if not asset:return []
 deno=WORK/'deno'/'deno'
 try:
  if not deno.exists():
   deno.parent.mkdir(parents=True,exist_ok=True)
   archive=deno.parent/'deno.zip'
   urllib.request.urlretrieve('https://github.com/denoland/deno/releases/latest/download/'+asset,archive)
   with zipfile.ZipFile(archive) as z:z.extract('deno',deno.parent)
   deno.chmod(0o755)
  subprocess.run([str(deno),'--version'],check=True,stdout=subprocess.DEVNULL,timeout=10)
  return ['--js-runtimes',f'deno:{deno}','--remote-components','ejs:github']
 except Exception as exc:
  print('Deno unavailable; yt-dlp fallback:',type(exc).__name__)
  return []
JS_ARGS=deno_args()

def run(args,timeout=300):
 try:
  p=subprocess.run(args,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=timeout,check=False)
  return p.returncode,p.stdout
 except subprocess.TimeoutExpired as exc:return 124,str(exc)

def info(path):
 code,out=run(['ffprobe','-v','error','-select_streams','v:0','-show_entries','stream=width,height,duration','-of','json',str(path)],30)
 if code:return None
 try:
  s=json.loads(out)['streams'][0]
  return {'width':int(s['width']),'height':int(s['height']),'duration_s':float(s.get('duration') or 0)}
 except (ValueError,IndexError,KeyError):return None

def obtain(row):
 cached=[p for p in (WORK/'videos').glob(row.video_id+'.*') if p.suffix in {'.mp4','.mkv','.webm'} and info(p)]
 if cached and info(cached[0])['height']==360:return cached[0],''
 failures=[]
 for fmt in ('18/best[height=360][ext=mp4]','bestvideo[height=360]+bestaudio/best[height=360]'):
  args=['yt-dlp','--no-playlist','--no-progress','--retries','2','--fragment-retries','2','--socket-timeout','20','-f',fmt,'--merge-output-format','mp4','-o',str(WORK/'videos'/(row.video_id+'.%(ext)s')),*JS_ARGS,row.source_url]
  code,out=run(args,300)
  matches=[p for p in (WORK/'videos').glob(row.video_id+'.*') if p.suffix in {'.mp4','.mkv','.webm'} and info(p) and info(p)['height']==360]
  if code==0 and matches:return matches[0],''
  failures.append(' '.join(out.splitlines()[-4:])[-500:])
  if 'Sign in to confirm' in out or 'not a bot' in out:break
 return None,' | '.join(failures)

source_paths={}
source_status=[]
reserved_creators=set(reserved.creator.map(normalize))
reserved_channel_ids=set(reserved.channel_id.dropna().astype(str)) if 'channel_id' in reserved else set()
for row in selected.itertuples(index=False):
 record={'video_id':row.video_id,'creator':row.creator,'status':'unavailable','error':''}
 print('Getting',row.video_id,row.creator)
 video,error=obtain(row)
 if video is None:
  record['error']=error;source_status.append(record);continue
 metadata_path=OUTPUT/'metadata'/(row.video_id+'.json')
 code,out=run(['yt-dlp','--skip-download','--dump-single-json','--no-warnings',*JS_ARGS,row.source_url],90)
 if code==0:
  try:
   raw=json.loads(out)
   metadata={key:raw.get(key) for key in ('id','title','channel','channel_id','duration','license','availability','webpage_url')}
   metadata_path.write_text(json.dumps(metadata,indent=2,ensure_ascii=False)+'\n')
   if (metadata['id']!=row.video_id or metadata['channel_id']!=row.channel_id or
       normalize(metadata['channel']) in reserved_creators or metadata['channel_id'] in reserved_channel_ids):
    record.update(status='identity_conflict',error='Video or observed channel conflicts with reserved sources')
    source_status.append(record);continue
  except (ValueError,TypeError):
   record.update(status='metadata_review',error='Could not parse observed channel metadata')
   source_status.append(record);continue
 source_paths[row.video_id]=video
 record['status']='ready';source_status.append(record)

print('Video sources ready:',len(source_paths),'of',len(selected))


In [ ]:
def make_pages(frame_root,page_root,start_s,fps):
 frames=sorted(frame_root.glob('*.jpg'))
 pages=0
 for start in range(0,len(frames),12):
  tiles=[]
  for index in range(start,min(start+12,len(frames))):
   frame=cv2.imread(str(frames[index]))
   if frame is None:continue
   sec=float(start_s)+index/fps
   mm=int(sec//60);ss=sec-mm*60
   cv2.rectangle(frame,(0,0),(104,26),(0,0,0),-1)
   cv2.putText(frame,f'{mm:02d}:{ss:04.1f}',(5,18),cv2.FONT_HERSHEY_SIMPLEX,.5,(0,255,255),1)
   tiles.append(frame)
  if not tiles:continue
  while len(tiles)<12:tiles.append(np.zeros_like(tiles[0]))
  page=np.vstack([np.hstack(tiles[i:i+4]) for i in (0,4,8)])
  cv2.imwrite(str(page_root/f'{pages:03d}.jpg'),page,[cv2.IMWRITE_JPEG_QUALITY,87])
  pages+=1
 return pages,len(frames)

def dense_pages(row,video):
 frame_root=WORK/'dense_frames'/row.review_id;frame_root.mkdir(parents=True,exist_ok=True)
 page_root=OUTPUT/'dense_contact_sheets'/row.review_id;page_root.mkdir(parents=True,exist_ok=True)
 for stale in frame_root.glob('*.jpg'):stale.unlink()
 duration=float(row.end_s-row.start_s)
 args=['ffmpeg','-hide_banner','-loglevel','error','-y','-ss',str(float(row.start_s)),'-i',str(video),'-t',str(duration),'-vf','fps=2,scale=320:180:force_original_aspect_ratio=decrease,pad=320:180:(ow-iw)/2:(oh-ih)/2:black','-q:v','3',str(frame_root/'%06d.jpg')]
 code,out=run(args,600)
 if code or not list(frame_root.glob('*.jpg')):return 0,0,'ffmpeg: '+out[-500:]
 pages,frames=make_pages(frame_root,page_root,float(row.start_s),2)
 return pages,frames,''

def replacement_pages(row,video):
 frame_root=WORK/'screen_frames'/row.video_id;frame_root.mkdir(parents=True,exist_ok=True)
 page_root=OUTPUT/'replacement_contact_sheets'/row.video_id;page_root.mkdir(parents=True,exist_ok=True)
 for stale in frame_root.glob('*.jpg'):stale.unlink()
 args=['ffmpeg','-hide_banner','-loglevel','error','-y','-i',str(video),'-vf','fps=1/5,scale=320:180:force_original_aspect_ratio=decrease,pad=320:180:(ow-iw)/2:(oh-ih)/2:black','-q:v','3',str(frame_root/'%06d.jpg')]
 code,out=run(args,900)
 if code or not list(frame_root.glob('*.jpg')):return 0,0,'ffmpeg: '+out[-500:]
 pages,frames=make_pages(frame_root,page_root,0,.2)
 return pages,frames,''

dense_results=[]
for row in targets.itertuples(index=False):
 entry={'review_id':row.review_id,'video_id':row.video_id,'target_hypothesis':row.target_hypothesis,'start_s':row.start_s,'end_s':row.end_s,'status':'unavailable','frames':0,'pages':0,'error':''}
 if row.video_id in source_paths:
  inf=info(source_paths[row.video_id])
  if inf and row.end_s<=inf['duration_s']+1:
   pages,frames,error=dense_pages(row,source_paths[row.video_id]);entry.update(status='evidence_ready' if pages else 'decode_failed',frames=frames,pages=pages,error=error)
  else:entry.update(status='duration_mismatch',error=str(inf))
 else:entry['error']='Source not ready; see source_status.csv'
 dense_results.append(entry);print(row.review_id,entry['status'],entry['pages'],'pages')

replacement_results=[]
for row in supplemental.itertuples(index=False):
 entry={'video_id':row.video_id,'creator':row.creator,'target_class':row.target_class,'status':'unavailable','frames':0,'pages':0,'error':''}
 if row.video_id in source_paths:
  pages,frames,error=replacement_pages(row,source_paths[row.video_id]);entry.update(status='evidence_ready' if pages else 'decode_failed',frames=frames,pages=pages,error=error)
 else:entry['error']='Source not ready; see source_status.csv'
 replacement_results.append(entry);print('replacement',row.video_id,entry['status'],entry['pages'],'pages')

pd.DataFrame(source_status).to_csv(OUTPUT/'source_status.csv',index=False)
dense_status=pd.DataFrame(dense_results);dense_status.to_csv(OUTPUT/'dense_review_status.csv',index=False)
replacement_status=pd.DataFrame(replacement_results);replacement_status.to_csv(OUTPUT/'replacement_screen_status.csv',index=False)
display(dense_status[['review_id','target_hypothesis','status','frames','pages']])
display(replacement_status[['creator','target_class','status','pages']])


Dense sheets and replacement screens are **evidence only**. They do not independently approve labels, boundaries, identity groups, source rights, or training use. Review every promoted interval and pose coverage before landmark extraction.


In [ ]:
for relative,name in (
 ('ml/data/source_balance_v2/dense_review_targets.csv','dense_review_targets.csv'),
 ('ml/data/source_balance_v2/supplemental_candidates.csv','supplemental_candidates.csv'),
 ('ml/data/source_balance_v2/screening_review.csv','screening_review.csv')):
 shutil.copy2(paths[relative],OUTPUT/name)
metadata={'notebook':'30b_colab_source_balance_v2_dense_and_replacement_screen.ipynb','runtime':'CPU','pinned_commit':PINNED_COMMIT,'dense_regions':len(targets),'dense_evidence_ready':int((dense_status.status=='evidence_ready').sum()),'replacement_sources':len(supplemental),'replacement_evidence_ready':int((replacement_status.status=='evidence_ready').sum()),'training_allowed':False,'raw_video_exported':False,'landmarks_exported':False}
(OUTPUT/'run_metadata.json').write_text(json.dumps(metadata,indent=2)+'\n')
checksum_lines=[]
for path in sorted(p for p in OUTPUT.rglob('*') if p.is_file()):checksum_lines.append(f'{sha256(path)}  {path.relative_to(OUTPUT)}')
(OUTPUT/'SHA256SUMS.txt').write_text('\n'.join(checksum_lines)+'\n')
archive=shutil.make_archive(str(OUTPUT),'zip',OUTPUT)
print('Created:',archive,round(Path(archive).stat().st_size/1e6,1),'MB')
files.download(archive)
